In [25]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, glob, os
import scipy.io as sio

## Objectives
1. QC stuff: set up preQC, do QC offline
2. Consolidate: load QC (and create dir with clean clusters), append columns for spikes, region, coordinates
3. Inspect & save

### variables

In [41]:
patient = 22

## 1. QC stuff

### create separate dirs for cluster-figs & pca/wave figs for easier QC

In [ ]:
# init dirs
osort_CL_dir = f'../../results/2025{patient}/osort_mat/figs_CL'
osort_allCL_dir = f'../../results/2025{patient}/osort_mat/figs_allCL'
os.makedirs(osort_CL_dir, exist_ok=True)
os.makedirs(osort_allCL_dir, exist_ok=True)

# parse existing dir
for file in glob.glob(f'../../results/2025{patient}/osort_mat/figs/5/*'):

    # grab individual CL figs
    if 'CL' in os.path.basename(file) and 'ALL' not in os.path.basename(file):
        
        dest = os.path.join(osort_CL_dir, os.path.basename(file))
        if not os.path.exists(dest):
            print(f'Copying {file} to {dest}')
            os.system(f'cp {file} {osort_CL_dir}/')

    # grab CL_ALL and PCA figs to check for mergers
    if 'WAVES' in os.path.basename(file):
        
        dest = os.path.join(osort_allCL_dir, os.path.basename(file))
        if not os.path.exists(dest):
            print(f'Copying {file} to {dest}')
            os.system(f'cp {file} {osort_allCL_dir}/')


### create preQC_df
note: clustID called unitID going forward

In [28]:
# init preQC_df 
preQC_df = pd.DataFrame(columns=['chanID', 'unitID'])
chanIDs, unitIDs = [], []

# parse CL fig dir
for file in glob.glob(f'../../results/2025{patient}/osort_mat/figs_CL/*'):
    chanIDs.append(int(os.path.basename(file).split('_')[0][1:]))
    unitIDs.append(int(os.path.basename(file).split('_')[2]))

# create df
preQC_df = pd.DataFrame({'chanID': chanIDs, 'unitID': unitIDs})
preQC_df = preQC_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)

# save
preQC_df_path = f'../../results/2025{patient}/records/preQC_pt{patient}.csv'
if not os.path.exists(preQC_df_path):
    print(f'Saving preQC_df to {preQC_df_path}')
    preQC_df.to_csv(preQC_df_path, index=False)

preQC_df

,chanID,unitID
0,193,31
1,193,810
2,193,883
3,193,902
4,193,929
...,...,...
82,208,1284
83,208,1301
84,208,1329
85,208,1336


## 2. Start to consolidate

### load QC

In [29]:
QC_df = pd.read_csv(f'../../results/2025{patient}/records/QC_pt{patient}.csv')

# drop noisy unit
dropped_unitIDs = QC_df[QC_df['keep'] != 1]['unitID'].astype(int).tolist()
dropped_unitIDs.extend([0, 1, 99999999])
QC_df = QC_df[~QC_df['unitID'].isin(dropped_unitIDs)].copy().reset_index(drop=True)

# copy clean unit figs from osort_CL_dir to data/units/clean/patient/
units_clean_pt_dir = f'../../data/units/clean/2025{patient}'
os.makedirs(units_clean_pt_dir, exist_ok=True)

for unitID in QC_df['unitID']:
    src = glob.glob(os.path.join(osort_CL_dir, f'*CL_{unitID}*.png'))[0]
    dest = os.path.join(units_clean_pt_dir, os.path.basename(src))
    if not os.path.exists(dest): os.system(f'cp {src} {dest}')

assert len(QC_df) == len(glob.glob(os.path.join(units_clean_pt_dir, '*.png')))

QC_df

,chanID,unitID,keep,notes
0,193,883,1.0,NaN
1,193,929,1.0,NaN
2,199,2436,1.0,FR fluctuation
3,199,2472,1.0,FR fluctuation
4,199,2477,1.0,FR fluctuation
5,200,889,1.0,NaN
6,201,2271,1.0,NaN
7,201,2283,1.0,NaN
8,204,1726,1.0,NaN


### helpers

In [30]:
def getunitID2spikes(unitIDs, spikes):
    ''' return dict with keys=unique units, and vals = list of corresponding spikes '''
    
    # initialize unit2spikes with keys as unique unit IDs and empty list as values
    unit2spikes = {}
    for unitID, spike in zip(unitIDs, spikes):

        if unitID in dropped_unitIDs: continue
        if unitID not in unit2spikes: unit2spikes[unitID] = [] # initialize

        unit2spikes[unitID].append(spike)

    return unit2spikes

### start creating neur/spikes df. First, add spike data from OSort mats.

In [31]:
samp_rate = 1000000
# columns
chanID_list, unitID_list, spikes_list, num_spikes_list, FR_list = [], [], [], [], []

# go through OSort mat files
for mat_file in glob.glob(f'../../results/2025{patient}/osort_mat/sorted_mats/5/*_sorted_new.mat'):

    # load channel mat
    chanMat = sio.loadmat(mat_file)
    chanID = int(os.path.basename(mat_file).split('_')[0][1:])

    # load vectors of spike times and corresponding units
    if chanMat['assignedNegative'].size == 0: continue
    spike_vector = chanMat['newTimestampsNegative'][0] # len = total n_spikes
    unit_vector = chanMat['assignedNegative'][0] # len = total n_spikes

    # create unit_vector => [spike_vector] for QCed units
    unit2spikes = getunitID2spikes(unit_vector, spike_vector)
    for unitID, unit_spike_list in unit2spikes.items():
        chanID_list.append(chanID)
        unitID_list.append(unitID)
        spikes = np.array(unit_spike_list) / samp_rate  # seconds
        spikes_list.append(spikes)
        num_spikes_list.append(len(spikes))
        FR_list.append(len(spikes) / (spikes[-1] - spikes[0]))

neur_df = pd.DataFrame({'chanID': chanID_list, 'unitID': unitID_list, 'spikes': spikes_list, 'num_spikes': num_spikes_list, 'FR': FR_list})
assert len(neur_df) == len(QC_df), f'Length mismatch: neur_df has {len(neur_df)} rows, QC_df has {len(QC_df)} rows'
neur_df


,chanID,unitID,spikes,num_spikes,FR
0,200,889,"[1.2737333333333334, 4.168333333333334, 5.4483...",3236,1.921668
1,201,2271,"[17.292166666666667, 17.33396666666667, 18.380...",4520,2.710302
2,201,2283,"[18.104666666666667, 25.1578, 32.396, 47.1109,...",2065,1.238960
3,199,2477,"[0.3642666666666667, 0.38356666666666667, 0.39...",16289,9.665839
4,199,2436,"[6.560433333333334, 12.252566666666668, 47.828...",2996,1.784500
5,199,2472,"[7.637333333333334, 16.9942, 40.48683333333334...",1177,0.702765
6,193,929,"[1.4766333333333335, 1.5566666666666666, 1.716...",7246,4.303382
7,193,883,"[3.3423000000000003, 4.025933333333334, 4.1788...",2431,1.446764
8,204,1726,"[17.384533333333337, 17.67366666666667, 17.679...",2360,1.420181


### add region column by mapping channel -> region (label)

In [32]:
chanMap = sio.loadmat(glob.glob(f'../../results/2025{patient}/records/*ChannelMap*.mat')[0])

# variable across patients
if patient in [12, 21]: channelMap = chanMap['ChannelMap1'].flatten()
elif patient == 18: channelMap = chanMap['ChannelMap2'].flatten()

labelMap = chanMap['LabelMap'].flatten() # contains region labels
labelMap = np.array([str(label.squeeze()) for label in labelMap])  # convert to str

nan_mask = ~np.isnan(channelMap)
channel2label = dict(zip(channelMap[nan_mask], labelMap[nan_mask]))

neur_df['region'] = neur_df['chanID'].map(channel2label).fillna(neur_df['chanID']).apply(lambda x: str(x))
neur_df

IndexError: boolean index did not match indexed array along axis 0; size of axis is 272 but size of corresponding boolean axis is 306

### add coordinates columns by mapping region->coords

In [ ]:
# cleaning function to handle nested arrays and bytes
def clean_entry(x):
    while isinstance(x, (np.ndarray, list)):
        x = x[0]
    if isinstance(x, (bytes, bytearray)):
        x = x.decode("utf-8", errors="ignore")
    return str(x)

at some point, figure out this code line by line

In [ ]:
# load
electrodeInfo = sio.loadmat(glob.glob(
                f'../../results/2025{patient}/records/*DI_Electrodes*.mat')[0])
ElecMapRaw   = pd.DataFrame(electrodeInfo['ElecMapRaw']) # region -> ID
ElecXYZRaw   = pd.DataFrame(electrodeInfo['ElecXYZRaw']) # ID -> coordinates
ElecAtlasRaw = pd.DataFrame(electrodeInfo['ElecAtlasRaw']) # atlas coords?

atlas_index = 0 # NMM

# clean
region_s = ElecMapRaw[0].apply(clean_entry)                    # Series of regions
atlas_s  = ElecAtlasRaw.iloc[:, atlas_index].apply(clean_entry)  # Series of atlas regions

# build small tables
region2id_df = pd.DataFrame({"region": region_s.values,
                             "ID": np.arange(len(region_s))})
id2xyz_df = ElecXYZRaw.reset_index().rename(columns={'index':'ID', 0:'x', 1:'y', 2:'z'})
# xyz2atlasRegions = pd.DataFrame({"ID": np.arange(len(atlas_s)),
#                                  "atlas_region": atlas_s.values})

# merge
final_neur_df = (neur_df
                .merge(region2id_df, on='region', how='left')
                .merge(id2xyz_df, on='ID', how='left')
                # .merge(xyz2atlasRegions, on='ID', how='left')
                )
final_neur_df = final_neur_df.drop(columns=['ID'])
final_neur_df['spikes'] = final_neur_df['spikes'].apply(lambda x: np.array(x))  # ensure arrays


### 3. inspect & save

In [ ]:
print('example neuron')
print(f'last 5 spikes (s): {final_neur_df["spikes"].iloc[0][-5:]}')
print(f'last 5 spikes (min): {final_neur_df["spikes"].iloc[0][-5:]/60}')

# add patient as first column
if 'patient' not in final_neur_df: final_neur_df.insert(0, 'patient', patient)

final_neur_df

example neuron
last 5 spikes (s): [1654.1982     1654.27323333 1654.47123333 1654.54063333 1655.09963333]
last 5 spikes (min): [27.56997    27.57122056 27.57452056 27.57567722 27.58499389]


,patient,chanID,unitID,spikes,num_spikes,FR,region,x,y,z
0,18,210,4424,"[0.21733333333333335, 0.40853333333333336, 0.5...",9533,5.760531,mRAHIP2,16.600004,-3.000006,-16.400003
1,18,210,4546,"[0.3117666666666667, 0.3276333333333334, 0.340...",6376,3.833812,mRAHIP2,16.600004,-3.000006,-16.400003
2,18,210,4457,"[0.8058666666666667, 11.403633333333334, 17.65...",5651,3.417072,mRAHIP2,16.600004,-3.000006,-16.400003
3,18,213,2153,"[0.6681666666666668, 0.9681333333333334, 2.171...",3999,2.402830,mRAHIP5,17.800004,-3.000006,-16.400003
4,18,213,2104,"[10.1952, 27.753466666666668, 47.0327333333333...",1163,0.703192,mRAHIP5,17.800004,-3.000006,-16.400003
5,18,200,833,"[1.5603333333333336, 3.720666666666667, 5.7518...",1301,0.788853,mROFC8,3.000004,44.199992,-10.400004
6,18,207,1988,"[0.25660000000000005, 0.5228666666666667, 0.72...",3825,2.299444,mRACC7,9.000004,27.399993,23.199994
7,18,207,2001,"[2.7072, 3.047266666666667, 3.3977333333333335...",5357,3.224241,mRACC7,9.000004,27.399993,23.199994
8,18,216,1079,"[2.503, 13.184033333333334, 28.113600000000005...",450,0.270778,mRAHIP8,19.000003,-3.000006,-16.400003
9,18,216,1110,"[2.805766666666667, 3.6407333333333334, 4.8141...",750,0.451857,mRAHIP8,19.000003,-3.000006,-16.400003


In [ ]:
# make dir
processed_data_dir = f'../../results/2025{patient}/records/processed_data'
os.makedirs(processed_data_dir, exist_ok=True)

# save file
parquet_path = os.path.join(processed_data_dir, 'df_neurs.parquet')
if os.path.exists(parquet_path):     print(f'File already exists, skipping: {parquet_path}')
else:     final_neur_df.to_parquet(parquet_path, index=False)
# final_neur_df.to_parquet(parquet_path, index=False)

File already exists, skipping: ../../results/202518/records/processed_data/df_neurs.parquet
